# Exploring sample station logs extracted

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col, from_unixtime,countDistinct

In [2]:
# Initialize a local Spark session
spark = SparkSession.builder \
    .appName("CitiBike-GBFS-Exploration") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

Spark Version: 4.1.2


## loading the json files

In [3]:
# Define the path to your mock Bronze zone
bronze_dir = "../data/bronze/real_time/"

# Load all station_information and station_status JSONs
# Spark will automatically read all files matching the pattern
raw_info_df = spark.read.json(f"{bronze_dir}*_station_information.json", multiLine=True)
raw_status_df = spark.read.json(f"{bronze_dir}*_station_status.json", multiLine=True)

## schemas:

In [4]:
print("Raw Info Schema:")
raw_info_df.printSchema()

Raw Info Schema:
root
 |-- data: struct (nullable = true)
 |    |-- stations: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- capacity: long (nullable = true)
 |    |    |    |-- eightd_has_key_dispenser: boolean (nullable = true)
 |    |    |    |-- eightd_station_services: array (nullable = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |-- electric_bike_surcharge_waiver: boolean (nullable = true)
 |    |    |    |-- external_id: string (nullable = true)
 |    |    |    |-- has_kiosk: boolean (nullable = true)
 |    |    |    |-- lat: double (nullable = true)
 |    |    |    |-- lon: double (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- region_id: string (nullable = true)
 |    |    |    |-- rental_methods: array (nullable = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |-- rental_uris: struct (nullable = true)
 |    |

In [5]:
print("Raw status Schema:")
raw_status_df.printSchema()

Raw status Schema:
root
 |-- data: struct (nullable = true)
 |    |-- stations: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- eightd_has_available_keys: boolean (nullable = true)
 |    |    |    |-- is_installed: long (nullable = true)
 |    |    |    |-- is_renting: long (nullable = true)
 |    |    |    |-- is_returning: long (nullable = true)
 |    |    |    |-- last_reported: long (nullable = true)
 |    |    |    |-- legacy_id: string (nullable = true)
 |    |    |    |-- num_bikes_available: long (nullable = true)
 |    |    |    |-- num_bikes_disabled: long (nullable = true)
 |    |    |    |-- num_docks_available: long (nullable = true)
 |    |    |    |-- num_docks_disabled: long (nullable = true)
 |    |    |    |-- num_ebikes_available: long (nullable = true)
 |    |    |    |-- num_scooters_available: long (nullable = true)
 |    |    |    |-- num_scooters_unavailable: long (nullable = true)
 |    |    |    |-- station_id: 

## flattening data

In [6]:
flat_info_df = raw_info_df.select(explode("data.stations").alias("s")).select("s.*")
flat_status_df = raw_status_df.select(explode("data.stations").alias("s")).select("s.*")
print(f"Total Flattened Info Rows: {flat_info_df.count()}")
print(f"Total Flattened Status Rows: {flat_status_df.count()}")

Total Flattened Info Rows: 2412
Total Flattened Status Rows: 7236


# GBFS Silver Layer Specification: dim_stations

**Source Feed:** `*_station_information.json`
**Target Table:** `silver.dim_stations`
**Update Strategy:** Slowly Changing Dimension (SCD Type 1 Upsert via ETag tracking)

### Grain Definition
One row represents one unique physical or virtual Citi Bike docking location in the network at its most recently known geographic state.

| Column Name | Triage Action | Target Silver Type | Analytical Rationale |
| :--- | :---: | :---: | :--- |
| **`station_id`** | **KEEP** | `STRING` | **Primary Key**. Globally unique identifier required to join high-velocity fact streams to static geography. |
| **`name`** | **KEEP** | `STRING` | Human-readable cross-street or landmark used for executive dashboard filtering. |
| **`lat`** | **KEEP** | `DOUBLE` | WGS84 Latitude. Required for spatial mapping of bike flows in Tableau (target precision: 6 decimal places). |
| **`lon`** | **KEEP** | `DOUBLE` | WGS84 Longitude. Required for spatial mapping (target precision: 6 decimal places). |
| **`capacity`** | **KEEP** | `INTEGER` | Total docking points installed. Serves as the fixed baseline denominator to calculate station saturation percentages. |
| **`region_id`** | **KEEP** | `STRING` | Identifies the political or neighborhood zone; critical for macro-level trip flow analysis. |
| `rental_uris` | DROP | `STRUCT` | Deep links (`android`, `ios`, `web`) designed to force-open native mobile apps; adds zero value to logistics. |
| `rental_methods` | DROP | `ARRAY` | Payment types accepted (e.g., Apple Pay, Key fobs). Commercial metadata that does not drive physical stock-outs. |
| `eightd_has_key_dispenser` | DROP | `BOOLEAN` | Vendor-specific hardware flag tracking physical key fob dispensers; unnecessary clutter. |
| `has_kiosk` | DROP | `BOOLEAN` | Hardware metadata indicating a payment screen; irrelevant to inventory rebalancing. |
| `eightd_station_services` | DROP | `ARRAY` | Legacy vendor service arrays; provides no active logistical signal. |
| `electric_bike_surcharge_waiver`| DROP | `BOOLEAN` | Financial/pricing rule flag; belongs in a pricing dimension, not a geographic asset table. |

> **Architect's Note for Pipeline Ingestion:** > Because this data updates at a low frequency, PySpark transformation must execute a `Window` function partitioned by `station_id` and ordered by the file snapshot timestamp descending to ensure the Silver layer only ever holds the single most recent version of a station's capacity and coordinates.

In [7]:
flat_info_df.toPandas().head(10)

,capacity,eightd_has_key_dispenser,eightd_station_services,electric_bike_surcharge_waiver,external_id,has_kiosk,lat,lon,name,region_id,rental_methods,rental_uris,short_name,station_id,station_type
0,23,False,[],False,66ddea99-0aca-11e7-82f6-3863bb44ef7c,True,40.666208,-73.981999,10 St & 7 Ave,71,"[KEY, CREDITCARD]","(https://bkn.lft.to/lastmile_qr_scan, https://...",3762.08,66ddea99-0aca-11e7-82f6-3863bb44ef7c,classic
1,78,False,[],False,06439006-11b6-44f0-8545-c9d39035f32a,True,40.712220,-74.010472,Vesey St & Church St,71,"[KEY, CREDITCARD]","(https://bkn.lft.to/lastmile_qr_scan, https://...",5216.06,06439006-11b6-44f0-8545-c9d39035f32a,classic
2,0,False,[],False,2206779404135604698,False,40.723710,-73.839590,70 Rd & 112 St,71,"[KEY, CREDITCARD]","(https://bkn.lft.to/lastmile_qr_scan, https://...",5543.05,2206779404135604698,classic
3,0,False,[],False,2206780889143294664,False,40.718940,-73.837700,Queens Blvd N & 75 Ave,71,"[KEY, CREDITCARD]","(https://bkn.lft.to/lastmile_qr_scan, https://...",5434.08,2206780889143294664,classic
4,0,False,[],False,2206781040135081610,False,40.716120,-73.837350,Austin St & 76 Ave,71,"[KEY, CREDITCARD]","(https://bkn.lft.to/lastmile_qr_scan, https://...",5324.03,2206781040135081610,classic
5,0,False,[],False,2206782424530885178,False,40.715320,-73.832260,Queens Blvd N & 78 Ave,71,"[KEY, CREDITCARD]","(https://bkn.lft.to/lastmile_qr_scan, https://...",5355.06,2206782424530885178,classic
6,39,False,[],False,0f45bcf6-7028-4584-a51e-4129847dbebc,True,40.732647,-73.990110,4 Ave & E 12 St,71,"[KEY, CREDITCARD]","(https://bkn.lft.to/lastmile_qr_scan, https://...",5788.15,0f45bcf6-7028-4584-a51e-4129847dbebc,classic
7,20,False,[],False,b35ba3c0-d3e8-4b1a-b63b-73a7bb518c9e,True,40.698000,-73.912700,Irving Ave & Palmetto St,71,"[KEY, CREDITCARD]","(https://bkn.lft.to/lastmile_qr_scan, https://...",4775.01,b35ba3c0-d3e8-4b1a-b63b-73a7bb518c9e,classic
8,25,False,[],False,457ea5d7-d643-4028-87db-46dee0b81555,True,40.725770,-73.941730,Kingsland Ave & Nassau Ave,71,"[KEY, CREDITCARD]","(https://bkn.lft.to/lastmile_qr_scan, https://...",5613.04,457ea5d7-d643-4028-87db-46dee0b81555,classic
9,24,False,[],False,1795146692060860976,True,40.823194,-73.887629,Longfellow Ave & Aldus St,71,"[KEY, CREDITCARD]","(https://bkn.lft.to/lastmile_qr_scan, https://...",7983.02,1795146692060860976,classic


# GBFS Silver Layer Specification: fact_station_status

**Source Feed:** `*_station_status.json`
**Target Table:** `silver.fact_station_status`
**Update Strategy:** High-Velocity Append (Micro-batch / Serverless Timer Trigger)

### Grain Definition
One row represents the exact physical inventory and operational status of a single station hardware unit at a specific logged point in time.

| Column Name | Triage Action | Target Silver Type | Analytical Rationale |
| :--- | :---: | :---: | :--- |
| **`station_id`** | **KEEP** | `STRING` | **Foreign Key**. Mandatory join key to link inventory facts back to `silver.dim_stations`. |
| **`last_reported`** | **KEEP** | `TIMESTAMP` | The exact UTC epoch timestamp the station hardware last transmitted its inventory to the backend. |
| **`num_bikes_available`** | **KEEP** | `INTEGER` | Dynamic inventory of classic pedal bikes physically docked and available for hire. |
| **`num_ebikes_available`** | **KEEP** | `INTEGER` | Dynamic inventory of pedal-assist electric bikes available for hire. |
| **`num_docks_available`** | **KEEP** | `INTEGER` | Critical metric to detect "Saturated Stations" (value = 0) where returning riders are blocked. |
| **`is_renting`** | **KEEP** | `INTEGER` | Boolean flag (`1`/`0`). Indicates whether the station is open for bike check-outs. |
| **`is_returning`** | **KEEP** | `INTEGER` | Boolean flag (`1`/`0`). Indicates whether the smart docks are unlocked to accept bike returns. |
| `num_scooters_available` | DROP | `INTEGER` | Standing e-scooter inventory; sits outside the logistical scope of bicycle fleet rebalancing. |
| `num_scooters_unavailable`| DROP | `INTEGER` | Broken/locked standing e-scooters; zero relevance to bike redistribution. |
| `legacy_id` | DROP | `STRING` | Deprecated system ID; redundant to the primary `station_id`. |
| `eightd_has_available_keys` | DROP | `BOOLEAN` | Vendor-level hardware check for physical kiosk keys. |
| `is_installed` | DROP | `INTEGER` | Street presence flag. Redundant; an uninstalled station is already captured via `is_renting == 0`. |
| `num_bikes_disabled` | DROP | `INTEGER` | Broken bike count. **Unreliable field**: vendors frequently omit or obfuscate disabled hardware counts. |
| `num_docks_disabled` | DROP | `INTEGER` | Broken dock count. Omitted for the same data-integrity reasons as disabled bikes. |

> **Logistical KPI Business Rule:** > When querying this table downstream in the Gold Layer to calculate **Stock-out Rate KPI** (stations completely empty of bikes), the filter must strictly be define as:
> `(num_bikes_available == 0) AND (is_renting == 1)`
> *Filtering purely for available bikes = 0 will accidentally count stations shut down for winter or street paving as logistical stock-out failures.*

In [8]:
flat_status_df.toPandas().head(10)

,eightd_has_available_keys,is_installed,is_renting,is_returning,last_reported,legacy_id,num_bikes_available,num_bikes_disabled,num_docks_available,num_docks_disabled,num_ebikes_available,num_scooters_available,num_scooters_unavailable,station_id
0,False,0,0,0,1781794028,4144,0,0,0,0,0,NaN,NaN,90b141b9-c39f-4a26-a32d-08c7d1474d52
1,False,0,0,0,86400,1795146692060860976,0,0,0,0,0,NaN,NaN,1795146692060860976
2,False,0,0,0,86400,3306,0,0,0,0,0,NaN,NaN,66ddea99-0aca-11e7-82f6-3863bb44ef7c
3,False,0,0,0,86400,4648,0,0,0,0,0,NaN,NaN,06439006-11b6-44f0-8545-c9d39035f32a
4,False,0,0,0,86400,2206779404135604698,0,0,0,0,0,NaN,NaN,2206779404135604698
5,False,0,0,0,86400,2206780889143294664,0,0,0,0,0,NaN,NaN,2206780889143294664
6,False,0,0,0,86400,2206781040135081610,0,0,0,0,0,NaN,NaN,2206781040135081610
7,False,0,0,0,86400,2206782424530885178,0,0,0,0,0,NaN,NaN,2206782424530885178
8,False,0,0,0,86400,3905,0,0,0,0,0,NaN,NaN,0f45bcf6-7028-4584-a51e-4129847dbebc
9,False,0,0,0,86400,3891,0,0,0,0,0,NaN,NaN,b35ba3c0-d3e8-4b1a-b63b-73a7bb518c9e


## droping noise columns

In [12]:

info_cols_to_drop = [
    "rental_uris",
    "rental_methods",
    "eightd_has_key_dispenser",
    "has_kiosk",
    "eightd_station_services",
    "electric_bike_surcharge_waiver"
]

status_cols_to_drop = [
    "num_scooters_available",
    "num_scooters_unavailable",
    "legacy_id",
    "eightd_has_available_keys",
    "is_installed",
    "num_bikes_disabled",
    "num_docks_disabled"
]

In [ ]:
silver_info_df = flat_info_df.drop(*info_cols_to_drop)
silver_status_df = flat_status_df.drop(*status_cols_to_drop)

In [11]:
print("=== Cleaned Silver Info Schema (dim_stations) ===")
silver_info_df.printSchema()

print("\n=== Cleaned Silver Status Schema (fact_station_status) ===")
silver_status_df.printSchema()

=== Cleaned Silver Info Schema (dim_stations) ===
root
 |-- capacity: long (nullable = true)
 |-- external_id: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- name: string (nullable = true)
 |-- region_id: string (nullable = true)
 |-- short_name: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- station_type: string (nullable = true)


=== Cleaned Silver Status Schema (fact_station_status) ===
root
 |-- is_renting: long (nullable = true)
 |-- is_returning: long (nullable = true)
 |-- last_reported: long (nullable = true)
 |-- num_bikes_available: long (nullable = true)
 |-- num_docks_available: long (nullable = true)
 |-- num_ebikes_available: long (nullable = true)
 |-- station_id: string (nullable = true)

